# Dataset formatting

This code will create a `dataset.csv` that contains a summary of the previously collected datasets stored in `../datasets/`. The resulting csv file will then be used for further data preprocessing.

In [54]:
# Imports

import os
import shutil as sh
from dataclasses import dataclass
import pandas as pd
import numpy as np
from PIL import Image

In [55]:
cwd: str = os.getcwd()
rel_datasets_path: str = "../datasets"
abs_dataset_path: str = os.path.join(cwd, rel_datasets_path)

def copy_dataset(dataset: str, path: str = abs_dataset_path) -> None:
    dir_path = os.path.join(path, dataset)
    dst_path = os.path.join(cwd, dataset)

    sh.copytree(dir_path, dst_path)

## CKplus dataset

This dataset is easy to handle since the files are already stored in csv format.

In [56]:
dataset: str = "CKplus"
copy_dataset(dataset)

In [57]:
# Change column labels to lowercase
file = os.listdir(dataset)[0]
filepath = os.path.join(dataset, file)

df = pd.read_csv(filepath)
df.columns = df.columns.str.lower()

In [58]:
# drop emotions Neutral and contempt since they are not needed
mask = df["emotion"] > 5
df = df.drop(df.loc[mask].index)
df = df.reset_index(drop = True)

In [59]:
# Drop the usage column. A split will later be created by the group
df = df.drop("usage", axis = 1)

In [60]:
# Each image will be converted to an image and saved to a new collection directory
collection_dir = os.path.join(cwd, "dataset")
os.mkdir(collection_dir)
num_rows = df.shape[0]

In [61]:
arr = []
images = df["pixels"].tolist()

for img in images:
    img = list(map(int, img.split()))
    arr.append(img)

arr = np.array(arr)

df = df.drop("pixels", axis = 1)

In [62]:
for i in range(num_rows):
    filename: str = f"CKPlus_{i + 1}.png"
    filepath = os.path.join(collection_dir, filename)

    img:np.ndarray = arr[i].reshape(48, 48)
    img = Image.fromarray(img.astype(np.uint8))
    img.save(filepath)

    df["filename"] = filename

In [63]:
df.to_csv("dataset.csv", index = False)

In [64]:
sh.rmtree(dataset)

## FERPlus

The next dataset to be converted will be FERPlus. The size of this dataset is 112 x 112.

In [65]:
emotion_map = {
    "angry": 0,
    "disgust": 1,
    "fear": 2,
    "happy": 3,
    "sad": 4,
    "suprise": 5,
}

image_size = 112

@dataclass
class Entry:
    emotion: int
    filename: str

type DataList = list[Entry]

dataset: str = "FERPlus"
copy_dataset(dataset)

### First Step

The first step will be to remove the subfolders containing the emotions we are not interested in

In [66]:
data_dir = os.path.join(cwd, dataset)
os.chdir(data_dir)
for dir in os.listdir(data_dir):
    os.chdir(dir)
    sub_dirs = os.listdir(os.getcwd()) # These are the emotion directories
    if "contempt" in sub_dirs:
        sh.rmtree("contempt")
    if "neutral" in sub_dirs:
        sh.rmtree("neutral")
    os.chdir(data_dir)

os.chdir(cwd)

### Next

I will create a new directory called `collection` with the same subdirs as the test, train and validation directories and I will 
add all images of an emotion into that directory

In [67]:
collection_dir = os.path.join(data_dir, "collection")
os.mkdir(collection_dir)

test_dir = os.listdir(data_dir)[0]
category_dir_1 = os.path.join(data_dir, test_dir)
categories = os.listdir(category_dir_1)

for category in categories:
    os.mkdir(os.path.join(collection_dir, category))

In [68]:
# Here we save all images into the collection dir

os.chdir(data_dir)

for dir in os.listdir(os.getcwd()):
    if dir == "collection":
        continue

    os.chdir(os.path.join(os.getcwd(), dir))

    for category in os.listdir(os.getcwd()):
        os.chdir(os.path.join(os.getcwd(), category))

        for file in os.listdir(os.getcwd()):
            sh.copy(file, os.path.join(collection_dir, category))
        
        os.chdir(os.path.join(os.getcwd(), ".."))
    
    os.chdir(data_dir)
os.chdir(cwd)

### Next

I will defin a dictionary to map written emotions to numbers. The numbers used are from the ck+ dataset. I will also define micellaneous datastructures and types that will be used late ron.

In [69]:
file_mappings: DataList = []

for subdir in os.listdir(collection_dir):
    sub_path = os.path.join(collection_dir, subdir)
    emotion = emotion_map[subdir]

    for i, file in enumerate(os.listdir(sub_path)):
        filename: str = f"FERPlus_{i}.png"

        filepath: str = os.path.join(cwd, "dataset")
        filepath = os.path.join(filepath, filename)

        sh.copy(os.path.join(sub_path, file), filepath)

        file_mappings.append(Entry(emotion=emotion, filename=filename))

In [70]:
df1 = pd.read_csv("dataset.csv")
df2 = pd.DataFrame(file_mappings)

combined = pd.concat([df1, df2], ignore_index = True)
combined.to_csv("dataset.csv", index = False)

In [71]:
sh.rmtree(dataset)

## AffectNet

Next I will add the AffectNet dataset

In [72]:
dataset: str = "AffectNet"
copy_dataset(dataset)